# MECHA in metafspm, 3: a pine needle, its water and its sugars over a day

MECHA's reactions branch simulates sucrose and starch in one pine needle section: photosynthesis in the mesophyll,
transport by diffusion and with the water, loading into the phloem, starch in the cells, and the sugars' osmotic
effect on the water. Its port, `MechaSolutes`, reproduces it on the same section
(`test/comparison/test_solutes.py`, `example/comparison/compare_solutes*.py`).

Here, in metafspm: a needle **organ** of three sections joined by their xylem and phloem, the organ's
transpiration leaving through the stomata of all of them, and the sugars coupled to the water both ways.

> **Developer tutorial 3** of ten:
> | | notebook | what it covers |
> |---|---|---|
> | 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
> | 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
> | 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section; a root partly out of contact, by position or by cells |
> | 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
> | 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
> | 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
> | 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |
> | 7 | `07_section_options.ipynb` | the section's hydraulic options: the barrier codes 0-11, φ-thickenings, aquaporins, plasmodesmata rules, walls, xylem pieces, tracheids, measured axial conductances |
> | 8 | `08_standard_problem.ipynb` | the standard problem's boundaries (soil sides, xylem and phloem pressures or flows, the elongation zone), the osmotic scenarios, the flows by layer |
> | 9 | `09_solute_options.ipynb` | the solutes' pathways, schemes and time steps, held concentrations and steady states, the coupled water and sugar |
>
> New to MECHA? Start with the user tutorials (`../user/01_first_run.ipynb`): they solve a root section and explain
> its results without the framework's details.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openalea.metafspm.coupling.composite_wrapper import CompositeModel
from openalea.metafspm.coupling.translator import Translator
from openalea.metafspm.scene.scene import Scene

from openalea.granap.metafspm.architectures import Needle
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.mecha.component import MechaHydraulics
from openalea.mecha.osmotic import MechaOsmoticPresets
from openalea.mecha.solutes import MechaSolutes

## 1. The plant model

Four components on one DataStructure, called in this order at each step:

| component | reads | writes |
|---|---|---|
| `GranapAnatomy` | the architecture | the sections' networks |
| `MechaOsmoticPresets` | the networks, the sucrose's osmotic potential | `osmotic_potential`, `reflection_coefficient` |
| `MechaHydraulics` | the networks, the osmotic terms, `leaf_transpiration` | `psi`, `flow` |
| `MechaSolutes` | the networks, the water's `flow`, `sucrose_source` | `sucrose`, `starch`, `sucrose_osmotic_potential` |

The organ's transpiration, `leaf_transpiration`, is an input at the **Organ** scale (one value per leaf organ, as a
transpiration model gives it). It is read when MECHA is built, so the model sets it on the organs first; MECHA puts
it at the organ's surface node, from which the stomatal Connections of all its sections lead to the pores.

In [2]:
class NeedleWaterAndSugars(CompositeModel):
    initiators = (Needle, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    def __init__(self, data_structure, time_step, transpiration=0.03):
        self.anatomy = GranapAnatomy(data_structure=data_structure, time_step=time_step)
        g = data_structure.mtg
        for organ in data_structure.entity_ids("Organ"):               # cm3 d-1 per leaf organ
            g.property("leaf_transpiration")[int(organ)] = transpiration
        self.osmotic = MechaOsmoticPresets(data_structure=data_structure, time_step=time_step)
        self.hydraulics = MechaHydraulics(data_structure=data_structure, time_step=time_step)
        self.solutes = MechaSolutes(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(
            translator_path=Translator(), components=(self.anatomy, self.osmotic, self.hydraulics, self.solutes))

    def run(self):
        self.anatomy()
        self.osmotic()
        self.hydraulics()
        self.solutes()

## 2. The scenario

Every component's parameters in one dict, as the reactions branch's needle has them: three leaf elements of 2 mm
(the `Needle` architecture), GRANAP's needle anatomy (seed 42, the reactions' section), the needle's wall and
membrane conductivities, the osmotic presets on every section with σ 0.7 (the presets are root rules: the
reactions branch applies them to the needle until a solute model gives its own baseline), the twig's xylem at
−5000 hPa at the needle's base, and the starch rates.

In [3]:
scenario = {
    "leaf_elements": 3, "leaf_element_length": 2e-3,
    "leaf_anatomy": {"preset": "for_needle", "seed": 42},
    "kw": 2.5634e-3, "kw_meso_hypo": 1e-16, "kmb": 3.6314e-4,
    "presets_sections": 2., "s_factor": 0.7,
    "collar_condition": 1., "collar_psi": -5000., "standard_solve": 0.,
    "k_starch_synthesis": 2.0, "k_starch_degradation": 1.0,
}
scene = Scene(pd.DataFrame([dict(plant="pine", model=NeedleWaterAndSugars, x=0., y=0., z=0., rotation=0.,
                                 scenario={"parameters": scenario, "transpiration": 0.03})]))
ds = scene.populations[0].data_structure
kind, path, owner = ds.get("kind"), ds.get("network_path"), ds.owner("SubOrgan")
print(f"{len(ds.entity_ids('SubOrgan'))} sections, {ds.n_nodes()} Compartments; "
      f"{int((path == 5).sum())} axial junctions; {int((kind == 3).sum())} leaf surface node; "
      f"{int((path == 6).sum())} stomatal Connections")

3 sections, 48586 Compartments; 1420 axial junctions; 1 leaf surface node; 12 stomatal Connections


## 3. The water at noon

One step at the noon transpiration: the organ's value at its surface node, through the stomatal Connections, out
of the pores of each section.

In [4]:
scene.run()
evaporation = ds.get("evaporation")
print(f"leaf_transpiration {ds.get('leaf_transpiration')[0]:.4f} cm³ d⁻¹ → surface node "
      f"{ds.get('surface_inflow')[kind == 3][0]:.4f} → stomatal Connections {ds.get('flow')[path == 6].sum():.4f} → "
      f"pores of each section " + ", ".join(f"{evaporation[owner == i].sum():.4f}" for i in range(3)))

leaf_transpiration 0.0300 cm³ d⁻¹ → surface node -0.0300 → stomatal Connections 0.0300 → pores of each section 0.0100, 0.0100, 0.0100


Across a membrane MECHA's flow is K (Δψ + σ Δπ), with σ 0.7: per tissue, the walls are under tension near the
twig's xylem pressure; the living cells hold turgor with their osmotic potential; the phloem, with the presets'
sieve tubes' −10000 hPa, is turgid.

In [5]:
psi, pi, tissue = ds.get("psi"), ds.get("osmotic_potential"), ds.get("tissue")
liquid = ds.get("vapour_node") == 0
groups = {name: (kind == 0) & (tissue == name) & liquid for name in
          ("xylem", "transfusion", "endodermis", "phloem", "Strasburger cell", "mesophyll", "epidermis")}
groups["walls and junctions"] = (kind > 0) & (kind < 3) & liquid
table = pd.DataFrame({"ψ (hPa)": {n: psi[m].mean() for n, m in groups.items()},
                      "π (hPa)": {n: pi[m].mean() for n, m in groups.items()}})
table["ψ + π (hPa)"] = table.sum(axis=1)
table.plot.bar(figsize=(11, 3.8), width=0.8, rot=20, title="the needle's potentials by tissue, at noon")
plt.show()

/tmp/ipykernel_1733691/2013410945.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. A day

Each hour, the forcings a transpiration model and a photosynthesis model would give: the organ's transpiration and
the mesophyll's sucrose source follow the light. In each section the sucrose moves radially to the sinks (the
phloem-loading Strasburger cells and the xylem); its transport along the needle in the phloem is not modelled (nor
in the reactions branch).

In [6]:
DT_DAYS = 1. / 24.
mesophyll = (kind == 0) & (tissue == "mesophyll") & liquid
light = lambda hour: max(0., np.sin(np.pi * (hour - 6.) / 12.))
history = []
for hour in range(24):
    ds.set("leaf_transpiration", [0.03 * max(light(hour + 0.5), 0.05)])
    ds.set("sucrose_source", np.where(mesophyll, ds.get("node_volume") * 5e-6 * light(hour + 0.5) / DT_DAYS, 0.))
    scene.run()
    c, s, V = ds.get("sucrose"), ds.get("starch"), ds.get("node_volume")
    held = (ds.get("liquid_node") > 0) & (ds.get("sucrose_sink") == 0)
    history.append({"transpiration": ds.get("evaporation").sum(),
                    "mesophyll ψ + π": ds.get("water_potential")[mesophyll].mean(),
                    "sucrose in the cells": (c * V)[held & (kind == 0)].sum(),
                    "sucrose in the walls": (c * V)[held & (kind > 0)].sum(),
                    "starch": (s * V)[held].sum(),
                    "produced": ds.get("sucrose_source").sum() * DT_DAYS})
    if hour == 11:
        noon = c.copy()
day = pd.DataFrame(history, index=range(1, 25))
day["produced"] = day["produced"].cumsum()
day["taken by the sinks"] = day["produced"] - day[["sucrose in the cells", "sucrose in the walls", "starch"]].sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), layout="constrained")
day[["transpiration"]].plot(ax=axes[0], title="the needle's transpiration (cm³ d⁻¹)", xlabel="hour")
day[["mesophyll ψ + π"]].plot(ax=axes[1], title="mesophyll water potential (hPa)", xlabel="hour")
(day[["sucrose in the cells", "sucrose in the walls", "starch", "taken by the sinks", "produced"]] * 1e9).plot(
    ax=axes[2], title="where the photosynthate goes (nmol)", xlabel="hour")
plt.show()

/tmp/ipykernel_1733691/2783852993.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
x, y = ds.get("section_x"), ds.get("section_y")
middle = (owner == 1) & liquid
cells = middle & (kind == 0)
fig, axes = plt.subplots(1, 3, figsize=(18, 4), layout="constrained")
for ax, values, where, title in ((axes[0], noon, middle, "sucrose at noon"),
                                 (axes[1], ds.get("sucrose"), middle, "sucrose at the end of the day"),
                                 (axes[2], ds.get("starch"), cells, "starch at the end of the day (cells)")):
    points = ax.scatter(x[where], y[where], c=values[where] * 1e6, s=0.5 if where is middle else 4)
    fig.colorbar(points, ax=ax, label="µmol cm⁻³", shrink=0.9)
    ax.set(aspect="equal", title=f"middle section: {title}", xticks=[], yticks=[])
plt.show()

/tmp/ipykernel_1733691/1037539531.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. The transfusion tracheids as xylem

The needle's transfusion tissue (cgroup 18) holds tracheids, dead water-filled conduits between the endodermis and
the vascular bundle. By default MECHA treats their membranes as a stele cell's; `tracheid_as_xylem` 1 (`develop`'s)
gives them the xylem vessels' (aquaporins × 10⁴ with barriers: no membrane resistance), without the xylem's boundary.
At the noon transpiration:

In [8]:
hydraulics = scene.populations[0].instance.hydraulics
ds.set("leaf_transpiration", [0.03])
rows = {}
for value, name in ((0., "as stele cells"), (1., "as xylem")):
    hydraulics.tracheid_as_xylem = value
    scene.run()
    psi = ds.get("psi")
    rows[name] = {f"{tissue} ψ (hPa)": psi[groups[tissue]].mean() for tissue in ("xylem", "transfusion", "mesophyll")}
hydraulics.tracheid_as_xylem = 0.
pd.DataFrame(rows).T

,xylem ψ (hPa),transfusion ψ (hPa),mesophyll ψ (hPa)
as stele cells,-5000.088086,-4997.851727,-2494.638104
as xylem,-5000.088086,-4997.895372,-2501.356181


The transfusion tissue was already within 3 hPa of the xylem: its membranes were not what limits the needle's water.
As conduits, its tracheids follow the xylem even more closely, and pass its tension on to the mesophyll, whose
pressure falls by about 7 hPa (0.3 %) for the same transpiration. The option matters where the transfusion tissue is
thick or its membranes' conductivity low (try `kaqp_stele_factor=0.01`).

Next: `04_full_plant.ipynb`, a whole plant in its soil.